# ⚾ Homework: Moneyball Part 2, Runs Allowed
In class we found that **on-base percentage** and **slugging percentage** drive **runs scored**. But the A's win games through **run differential** (runs scored − runs allowed), so preventing runs matters just as much.

In this homework you'll build a regression model for **runs allowed**.

Run the setup cells below first. They load the same data and model-report helper we used in class.

## 🧰 Setup: load our tools

In [2]:
import pandas as pd
from pathlib import Path

# statsmodels builds our regression models
import statsmodels.api as sm
from statsmodels.tools.eval_measures import rmse

# sklearn scores our models and splits data into training and testing sets
from sklearn import metrics
from sklearn.model_selection import train_test_split

## 🔎 Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Renames the columns: abbreviations are spelled out using a dictionary, and every label is made lower case with underscores for spaces (`RS` becomes `runs_scored`).
3. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
4. Creates **run differential**: `runs_scored − runs_allowed`. It's stored as `runs_diff`.

In [3]:
# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ['data/baseball.csv', '../data/baseball.csv'] if Path(p).exists())
df = pd.read_csv(data_path)

In [4]:
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()

In [5]:
# Time machine: keep only seasons before Moneyball (2002)
df = df[df['year'] < 2002].reset_index(drop=True)

In [6]:
# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

df.head()

,team,league,year,runs_scored,runs_allowed,wins,on_base_percentage,slugging_percentage,batting_average,playoffs,rank_season,rank_playoffs,games,opponent_obp,opponent_slg,runs_diff
0,ANA,AL,2001,691,730,75,0.327,0.405,0.261,0,NaN,NaN,162,0.331,0.412,-39
1,ARI,NL,2001,818,677,92,0.341,0.442,0.267,1,5.0,1.0,162,0.311,0.404,141
2,ATL,NL,2001,729,643,88,0.324,0.412,0.260,1,7.0,3.0,162,0.314,0.384,86
3,BAL,AL,2001,687,829,63,0.319,0.380,0.248,0,NaN,NaN,162,0.337,0.439,-142
4,BOS,AL,2001,772,745,82,0.334,0.439,0.266,0,NaN,NaN,161,0.329,0.393,27


#### Data Dictionary

| **Column**              | **Original** | **Description**                                          |
| ----------------------- | ------------ | -------------------------------------------------------- |
| **team**                | Team         | Name of the team                                         |
| **league**              | League       | Name of the league                                       |
| **year**                | Year         | Year of the season                                       |
| **runs_scored**         | RS           | Runs scored by the team                                  |
| **runs_allowed**        | RA           | Runs allowed by the team                                 |
| **wins**                | W            | Number of wins                                           |
| **on_base_percentage**  | OBP          | On-Base Percentage                                       |
| **slugging_percentage** | SLG          | Slugging Percentage                                      |
| **batting_average**     | BA           | Batting Average                                          |
| **playoffs**            | Playoffs     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **rank_season**         | RankSeason   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **rank_playoffs**       | RankPlayoffs | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **games**               | G            | Number of games played                                   |
| **opponent_obp**        | OOBP         | Opponent On-Base Percentage                              |
| **opponent_slg**        | OSLG         | Opponent Slugging Percentage                             |
| **runs_diff**           | (new)        | Runs scored minus runs allowed (`runs_scored - runs_allowed`) |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

## 🛠️ Helper function
**`print_model_report`** is the same helper we used in class. It prints the coefficients and p-values, then R², RMSE, and the average of the dependent variable. By default it scores the model on the data it was trained on. Pass in different `X` and `y` (like a test set) to score it on that data instead.

In [7]:
def print_model_report(model, X=None, y=None):
    # Coefficient and p-value for each variable
    report = pd.DataFrame({'coefficient': model.params, 'p_value': model.pvalues})
    print(report.round(4))
    print()

    # By default, score the model on the data it was trained on
    # (model.model.exog holds the X values; model.model.endog holds the actual y values)
    if X is None:
        X = model.model.exog
        y = model.model.endog

    # How well the model fits
    y_pred = model.predict(X)
    print('R-squared:', round(metrics.r2_score(y, y_pred), 4))
    print('RMSE:', round(rmse(y, y_pred), 2))
    print(f'Average {model.model.endog_names}:', round(y.mean(), 2))

---
## 🧤 Your task: How do we ALLOW fewer runs?
Winning is about **run differential**, so preventing runs counts just as much as scoring them. Let's flip the question around and use the **opponents'** hitting stats against our pitchers:

* **Dependent variable:** `runs_allowed`
* **Try predictors:** `opponent_obp` (opponents' on-base %) and `opponent_slg` (opponents' slugging %)

⚠️ Check for nulls first (`df.info()`): `opponent_obp` and `opponent_slg` are **missing for most of the older seasons**. Drop those rows first (e.g. `df.dropna(subset=['opponent_obp', 'opponent_slg'])`), or the model will error out.

**Your turn:** build and evaluate this model by following the steps below. Then decide: do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model from class (R² ≈ 0.93 using `on_base_percentage` and `slugging_percentage`)?

### 1. Check for nulls and drop missing rows

In [42]:
# Check for nulls, then drop the rows that are missing opponent_obp or opponent_slg
df.isna().sum()
df = df.dropna(subset=["opponent_obp", "opponent_slg"])

df.isna().sum()


team                    0
league                  0
year                    0
runs_scored             0
runs_allowed            0
wins                    0
on_base_percentage      0
slugging_percentage     0
batting_average         0
playoffs                0
rank_season            66
rank_playoffs          66
games                   0
opponent_obp            0
opponent_slg            0
runs_diff               0
dtype: int64

### 2. Define X and y
Add a constant to `X` (`sm.add_constant`) so the model can learn an intercept.

In [43]:
# Define X (the predictors, plus a constant) and y (runs_allowed)
X = pd.get_dummies(df, columns=["rank_season", "rank_playoffs"])
X.drop(columns=["team", "league", "runs_allowed", "runs_diff"], inplace=True)
X = sm.add_constant(X)

y = df["runs_allowed"]

### 3. Build the model on all the data
Fit the model with `sm.OLS(y, X).fit()`, then print the results with `print_model_report`.

In [44]:
X.head()

,const,year,runs_scored,wins,on_base_percentage,slugging_percentage,batting_average,playoffs,games,opponent_obp,...,rank_season_2.0,rank_season_3.0,rank_season_4.0,rank_season_5.0,rank_season_6.0,rank_season_7.0,rank_playoffs_1.0,rank_playoffs_2.0,rank_playoffs_3.0,rank_playoffs_4.0
0,1.0,2001,691,75,0.327,0.405,0.261,0,162,0.331,...,False,False,False,False,False,False,False,False,False,False
1,1.0,2001,818,92,0.341,0.442,0.267,1,162,0.311,...,False,False,False,True,False,False,True,False,False,False
2,1.0,2001,729,88,0.324,0.412,0.260,1,162,0.314,...,False,False,False,False,False,True,False,False,True,False
3,1.0,2001,687,63,0.319,0.380,0.248,0,162,0.337,...,False,False,False,False,False,False,False,False,False,False
4,1.0,2001,772,82,0.334,0.439,0.266,0,161,0.329,...,False,False,False,False,False,False,False,False,False,False


In [52]:
# Build the model on all the data and print the model report
model = sm.OLS(y, X).fit()
print_model_report(model)

                     coefficient  p_value
const                  3241.0611   0.6812
year                     -1.5732   0.6852
runs_scored               0.2307   0.0727
wins                     -3.0561   0.0002
on_base_percentage      319.5161   0.6027
slugging_percentage     155.2072   0.5190
batting_average        -383.2954   0.4101
playoffs                  5.8276   0.4434
games                    -2.7142   0.6775
opponent_obp           1882.9721   0.0000
opponent_slg           1155.5528   0.0000
rank_season_1.0           0.4808   0.9733
rank_season_2.0          -4.4216   0.7344
rank_season_3.0           7.1099   0.7299
rank_season_4.0         -17.5870   0.1460
rank_season_5.0           9.7632   0.5887
rank_season_6.0          28.2939   0.1005
rank_season_7.0         -17.8115   0.4743
rank_playoffs_1.0        -8.7202   0.6143
rank_playoffs_2.0        16.3458   0.3451
rank_playoffs_3.0        -0.4663   0.9722
rank_playoffs_4.0        -1.3317   0.9060

R-squared: 0.9372
RMSE: 20.78
Ave

C:\Users\11791\AppData\Local\Temp\ipykernel_5076\2656263303.py:2: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y, X).fit()


### 4. Evaluate on a train/test split
Split the data with `train_test_split(X, y, test_size=0.2, random_state=42)` so everyone gets the same numbers. Fit a model on the training data, then score it on the testing data with `print_model_report(model, X_test, y_test)`.

In [ ]:
# Split the data, build the model on the training data, and score it on the testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = sm.OLS(y_train, X_train).fit()

print_model_report(model, X_test, y_test)

                     coefficient  p_value
const                 -3105.5295   0.7316
year                      1.4796   0.7379
runs_scored               0.1772   0.2380
wins                     -2.7179   0.0024
on_base_percentage      541.6140   0.4328
slugging_percentage     151.8155   0.6112
batting_average        -309.4573   0.5865
playoffs                  6.9983   0.4179
games                    -2.3324   0.7509
opponent_obp           2292.6904   0.0000
opponent_slg           1066.2553   0.0003
rank_season_1.0          19.6257   0.3452
rank_season_2.0          -8.4107   0.6101
rank_season_3.0          11.0347   0.6057
rank_season_4.0         -23.3671   0.1141
rank_season_5.0           6.6820   0.7212
rank_season_6.0          21.2317   0.2307
rank_season_7.0         -19.7978   0.4371
rank_playoffs_1.0        -8.9674   0.6119
rank_playoffs_2.0         6.7450   0.7214
rank_playoffs_3.0         1.6813   0.9108
rank_playoffs_4.0         7.5393   0.6002

R-squared: 0.8807
RMSE: 27.79
Ave

C:\Users\11791\AppData\Local\Temp\ipykernel_5076\1760998951.py:4: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = sm.OLS(y_train, X_train).fit()


In [56]:
print_model_report(model, X, y)

                     coefficient  p_value
const                 -3105.5295   0.7316
year                      1.4796   0.7379
runs_scored               0.1772   0.2380
wins                     -2.7179   0.0024
on_base_percentage      541.6140   0.4328
slugging_percentage     151.8155   0.6112
batting_average        -309.4573   0.5865
playoffs                  6.9983   0.4179
games                    -2.3324   0.7509
opponent_obp           2292.6904   0.0000
opponent_slg           1066.2553   0.0003
rank_season_1.0          19.6257   0.3452
rank_season_2.0          -8.4107   0.6101
rank_season_3.0          11.0347   0.6057
rank_season_4.0         -23.3671   0.1141
rank_season_5.0           6.6820   0.7212
rank_season_6.0          21.2317   0.2307
rank_season_7.0         -19.7978   0.4371
rank_playoffs_1.0        -8.9674   0.6119
rank_playoffs_2.0         6.7450   0.7214
rank_playoffs_3.0         1.6813   0.9108
rank_playoffs_4.0         7.5393   0.6002

R-squared: 0.9317
RMSE: 21.67
Ave

### 5. Interpret your results
✍️ **Answer here:** Do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model? How does the testing score compare with the score on all the data?

Looking at the data, it makes sense because our result is the total runs-scored model and the lowest p-value (what affects the result the most) is opponent_obp (how often do they reach base), opponent_slg (how hard the ball is hit).
The OBP and he SLG value makes sense since:

OBP: the more base they reach, the more runs scored
SLG: the harder they hit the ball, the higher chance hitting out, the more runs scored

These 2 does makes sense on how it'll affect the runs_scored. 

Overall the R^2 is 0.88 which means 88% of the data/inputs (total) affects the output (runs_allowed). RSME is at 27.79 which means that 27.79 of the run_allowed is predicted incorrectly out of the ~817 total which is only 3% error which is decent.

Comparing the score to all the data is 0.02% error which is better which makes sense since it's all data including the trained data